## RAG pipelines - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [7]:

# Data ingestion
# Read all PDF files inside a directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []

    # Convert directory string to Path
    pdf_dir = Path(pdf_directory)

    # Get all PDF files
    pdf_files = list(pdf_dir.glob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:

        print(f"\nProcessing: {pdf_file.name}")

        try:
            # Load PDF
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            # Add documents to list
            all_documents.extend(documents)

            print(f"Loaded {len(documents)} pages")

        except Exception as err:
            print(f"Error: {err}")

    return all_documents


# Process all PDFs
all_pdf_documents = process_all_pdfs("../data/pdf")

print(f"\nTotal pages loaded: {len(all_pdf_documents)}")

Found 6 PDF files to process

Processing: attention.pdf
Loaded 1 pages

Processing: embeddings.pdf
Loaded 1 pages

Processing: rag.pdf
Loaded 1 pages

Processing: vector_database.pdf
Loaded 1 pages

Processing: embeddings -deep.pdf
Loaded 1 pages

Processing: proposal.pdf
Loaded 1 pages

Total pages loaded: 6


In [9]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-20T00:47:24+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-20T00:47:24+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/attention.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content="Attention\nDefinition\nAttention is a technique that lets a model decide which parts of the input to focus on when producing\nan output, instead of treating every word equally.\nKey idea\nNot all words in a sentence matter equally for understanding a given word. Attention assigns a\nweight to every other word, showing how much it should be 'attended to' when processing the\ncurrent word.\nQuery, Key, Value\n\x7f\nQuery (Q): what the current word is looking for.\n\x7f\nKey (K): what each word offers, used to match against the query.\n\x7f\nVa

In [10]:
# TExt splitting get into chunks

def split_documents(documents, chunk_size=1000,chunk_overlap=200):
    """split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents=documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks") 

    # show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}")
        print(f"Metadata: {split_docs[0].metadata}")  

    return split_docs    

In [11]:
chunks = split_documents(all_pdf_documents)
chunks

Split 6 documents into 11 chunks

Example chunk:
Content: Attention
Definition
Attention is a technique that lets a model decide which parts of the input to focus on when producing
an output, instead of treating every word equally.
Key idea
Not all words in 
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-20T00:47:24+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-20T00:47:24+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/attention.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-20T00:47:24+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-20T00:47:24+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/attention.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content="Attention\nDefinition\nAttention is a technique that lets a model decide which parts of the input to focus on when producing\nan output, instead of treating every word equally.\nKey idea\nNot all words in a sentence matter equally for understanding a given word. Attention assigns a\nweight to every other word, showing how much it should be 'attended to' when processing the\ncurrent word.\nQuery, Key, Value\n\x7f\nQuery (Q): what the current word is looking for.\n\x7f\nKey (K): what each word offers, used to match against the query.\n\x7f\nVa

### Embeddings and vector store db

In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
class EmbeddingManager:
    """Handles document embeddings generation using SentenceTransformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager
        Args:
            model_name : HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as err:
            print(f"Error Loading model : {self.model_name}: {err}")
            raise    

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
            Args:
            texts: List of text strings to embed
            returns:
            numpy array of embeddings with shape (len(texts),embedding_dim)
        """    

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

    def get_embedding_function(self) -> int:
        """Get the embeddings dimension of the model"""
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()

## initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

        

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1611.54it/s]


Model loaded successfully. Embedding dimension: 384


/tmp/ipykernel_11212/3296859018.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


## VectorStore

In [22]:
import os
import uuid
from typing import List, Any

import chromadb
import numpy as np


class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""

        try:
            os.makedirs(self.persist_directory, exist_ok=True)

            # Create persistent ChromaDB client
            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                }
            )

            print(
                f"Vector store initialized: {self.collection_name}"
            )

            print(
                f"Existing documents: {self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """Add documents and embeddings to ChromaDB."""

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        print(
            f"Adding {len(documents)} documents to vector store..."
        )

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"

            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)

            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            # IMPORTANT
            metadatas.append(metadata)

            # Document text
            documents_text.append(
                doc.page_content
            )

            # IMPORTANT: individual embedding
            embeddings_list.append(
                embedding.tolist()
            )

        # Add everything to ChromaDB
        try:

            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Successfully added {len(documents)} documents"
            )

            print(
                f"Total documents: {self.collection.count()}"
            )

        except Exception as e:
            print(f"Error adding documents: {e}")
            raise
vector_store = VectorStore()
vector_store        

Vector store initialized: pdf_documents
Existing documents: 0


In [15]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-20T00:47:24+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-20T00:47:24+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/attention.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content="Attention\nDefinition\nAttention is a technique that lets a model decide which parts of the input to focus on when producing\nan output, instead of treating every word equally.\nKey idea\nNot all words in a sentence matter equally for understanding a given word. Attention assigns a\nweight to every other word, showing how much it should be 'attended to' when processing the\ncurrent word.\nQuery, Key, Value\n\x7f\nQuery (Q): what the current word is looking for.\n\x7f\nKey (K): what each word offers, used to match against the query.\n\x7f\nVa

In [ ]:
# convert the text to embeddings
texts = [doc.page_content for doc in chunks]
texts

["Attention\nDefinition\nAttention is a technique that lets a model decide which parts of the input to focus on when producing\nan output, instead of treating every word equally.\nKey idea\nNot all words in a sentence matter equally for understanding a given word. Attention assigns a\nweight to every other word, showing how much it should be 'attended to' when processing the\ncurrent word.\nQuery, Key, Value\n\x7f\nQuery (Q): what the current word is looking for.\n\x7f\nKey (K): what each word offers, used to match against the query.\n\x7f\nValue (V): the actual information carried by each word, combined based on the match.\nSimple example\nIn the sentence 'The cat sat because it was tired', the word 'it' should pay high attention to 'cat' and\nlow attention to 'sat' or 'tired', so the model correctly understands that 'it' refers to the cat.\nUse in RAG\nInside the language model that generates the final answer, attention helps it focus on the most",
 'Use in RAG\nInside the language m

In [23]:
# Generate embeddings
embeddings = embedding_manager.generate_embeddings(
    texts=texts
)

# Store in vector database
vector_store.add_documents(
    chunks,
    embeddings
)

Generating embeddings for 11 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.02it/s]

Generated embeddings with shape: (11, 384)
Adding 11 documents to vector store...
Successfully added 11 documents
Total documents: 11
